In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Chaos

By now we're pretty good at predicting the future.  Give us the forces and a starting state, and Verlet will tell you where a pendulum, a molecule, or a planet will be, and it'll keep the energy honest while it does.

Today we'll meet a machine that breaks our crystal ball.  It's just two pendulums and a twisty rod.  There's no randomness anywhere: the same start always gives exactly the same motion.  And yet, if you change where you start by an amount smaller than an atom, the motion soon becomes *completely* different.  That's **chaos**, sometimes called the *butterfly effect*.

We'll see:
 - a new conservative system with *two* angles, built on our `ConservativeSystem`,
 - what it does for small swings, which is beautifully predictable,
 - what it does for big swings, which isn't,
 - and how to put a number on *how fast* predictability runs out.

## Our Tools So Far

Here's what we need from last time, copied over without changes.

In [ ]:
class FirstOrderODE:
    '''
    A first-order ODE specifies the rate of change of a state y(t)
    in terms of
     - the state itself, y
     - the time, t.

    The generic form for an ODE is

    dy
    --  = f(t, y)
    dt

    '''

    def f(self, t, y):
        raise NotImplementedError('The generic ODE has an unspecified slope function f.')

    def exact_solution(self, t, *args):
        raise NotImplementedError('The exact solution is not known (or at least not provided).')

In [ ]:
class SecondOrderODE(FirstOrderODE):
    '''
    A second-order ODE specifies the acceleration

    d^2x
    ----  = a(t, x, v)
    dt^2

    where v = dx/dt is the velocity.  Treating the state as y = (x, v) turns it into
    a first-order ODE,

    d  ( x )   (    v     )
    -- (   ) = (          )
    dt ( v )   ( a(t,x,v) )

    so a subclass only needs to provide the acceleration.
    '''

    def acceleration(self, t, x, v):
        raise NotImplementedError('The generic second-order ODE has an unspecified acceleration.')

    def f(self, t, y):
        # Unpack the state into its two components,
        x, v = y
        # and pack their rates of change back up the same way.
        return np.array([v, self.acceleration(t, x, v)])

In [ ]:
class FirstOrderODEIntegrator:
    '''
    An integrator is an algorithm for producing numerical solutions to ODEs.
    '''

    def __init__(self, ODE):

        self.ODE = ODE

    def starting_from(self, t_0, y_0):
        raise NotImplementedError('No integration algorithm has been specified.')

In [ ]:
class VelocityVerlet(FirstOrderODEIntegrator):
    '''
    Velocity Verlet splits each step into a half kick, a drift, and another half kick,

    v_{n+1/2} = v_n       + 1/2 a(t_n, x_n) Δt
    x_{n+1}   = x_n       + v_{n+1/2} Δt
    v_{n+1}   = v_{n+1/2} + 1/2 a(t_{n+1}, x_{n+1}) Δt

    It needs the ODE's acceleration, so it only works on second-order ODEs.
    '''

    def __init__(self, ODE, Delta_t, steps):

        self.Delta_t = Delta_t
        self.steps = steps

        super().__init__(ODE)

    def starting_from(self, t_0, y_0):

        # Populate all the t values
        t = t_0 + self.Delta_t * np.arange(self.steps+1)
        # and allocate an array with room for a whole state at each time.
        y = np.zeros((self.steps+1,) + np.shape(y_0))

        # The user picked a starting state
        y[0] = y_0

        for step in range(self.steps):
            x, v = y[step]
            # (acceleration also wants a velocity; we hand it the newest one we have.
            #  For forces that depend only on position it doesn't matter.)
            v_half = v      + 0.5 * self.ODE.acceleration(t[step],   x,     v)      * self.Delta_t   # half kick
            x_new  = x      +       v_half                                           * self.Delta_t   # drift
            v_new  = v_half + 0.5 * self.ODE.acceleration(t[step+1], x_new, v_half) * self.Delta_t   # half kick
            y[step+1] = np.array([x_new, v_new])

        return t, y

In [ ]:
class ConservativeSystem(SecondOrderODE):
    '''
    A conservative system has a potential energy V(x), and its acceleration is

    a = -V'(x)

    which doesn't depend on the velocity or the time.  Its energy

    E = 1/2 v^2 + V(x)

    is conserved.  A subclass provides the potential and the acceleration (minus its slope!).
    Everything is per unit mass.
    '''

    def potential(self, x):
        raise NotImplementedError('The generic conservative system has an unspecified potential.')

    def energy(self, x, v):
        return 0.5 * v**2 + self.potential(x)

# Two Pendulums on a Twisty Axle

Here's the machine.  A thin steel rod, the *axle*, runs horizontally from front to back.  Bearings at its two ends let it turn freely.  Two pendulums are clamped tightly onto the axle, one near the front and one near the back.  Each pendulum swings in its own vertical plane, and the two planes are parallel.

 - If the pendulums swing *together*, the whole axle just turns as one piece.  Nothing twists.
 - If they swing *differently*, the stretch of axle between the two clamps gets twisted, by the difference of their angles, $\theta_1 - \theta_2$.

A twisted rod is a spring!  Twist it and it twists back, with a torque proportional to how much it's twisted,
\begin{align}
    \tau_\text{twist} &= -\kappa \times (\text{twist}).
\end{align}
That's Hooke's law for twisting, and $\kappa$ is the *torsion constant*, which says how stiff the rod is.  A car's anti-roll bar works exactly this way: it's a torsion bar connecting the left and right wheels, so it only pushes back when one wheel rises more than the other.

Let's draw it.  We'll paint stripes along the axle, so we can see it twist.  We'll want the same drawing again for a movie later, so the drawing code is in two helper functions: one for the parts that never move (the axle's tube and its bearings) and one for the pendulums and the stripes, which hands back a function that moves them to any pair of angles.  (The details are just matplotlib; feel free to skim them.)

In [ ]:
CLAMP = np.array([0.0, 1.0])     # where each pendulum is clamped along the axle (the y direction)
AXLE_ENDS = (-0.3, 1.3)          # where the bearings hold the axle
RADIUS = 0.12                    # the axle's radius (exaggerated so we can see it)
ELEV, AZIM = 20, -35             # the camera angle, in degrees

def draw_axle(ax, top=0.3):
    '''
    Draws the parts of the machine that never move: the axle's tube and the bearings that hold it.
    Also sets up the camera.
    '''
    y = np.linspace(*AXLE_ENDS, 80)
    around = np.linspace(0, 2*np.pi, 40)
    Y, A = np.meshgrid(y, around)
    # matplotlib's 3D depth sorting would paint the tube over the stripes, so we choose the drawing order ourselves.
    ax.computed_zorder = False
    ax.plot_surface(RADIUS * np.sin(A), Y, -RADIUS * np.cos(A), color='lightgray', alpha=0.6, linewidth=0, shade=False, zorder=1)
    for end in AXLE_ENDS:
        ax.plot([0, 0], [end, end], [-RADIUS, -0.6], color='black', linewidth=5, solid_capstyle='butt', zorder=0)

    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(AXLE_ENDS[0] - 0.1, AXLE_ENDS[1] + 0.1)
    ax.set_zlim(-1.2, top)
    ax.set_aspect('equal')
    ax.view_init(elev=ELEV, azim=AZIM)
    ax.set_box_aspect(None, zoom=1.5)
    ax.set_axis_off()

def draw_pendulums(ax, ell=(1.0, 1.0), alpha=1.0, color='C0', stripes=12):
    '''
    Draws the two pendulums, and (if stripes > 0) stripes painted on the axle that twist along with them.
    Returns a function that moves everything to new angles theta = (θ_1, θ_2).
    '''
    ell = np.array(ell)
    rods = [ax.plot([], [], [], color='black', linewidth=2, alpha=alpha, zorder=3)[0] for _ in range(2)]
    bobs = ax.plot([], [], [], marker='o', markersize=16, color=color, alpha=alpha, linestyle='none', zorder=4)[0]
    stripe_lines = [ax.plot([], [], [], color=('C3', 'dimgray')[k % 2], linewidth=4, zorder=2)[0] for k in range(stripes)]

    y = np.linspace(*AXLE_ENDS, 200)
    # Which way the camera is looking from, so we can hide stripes on the far side of the axle.
    elev, azim = np.radians(ELEV), np.radians(AZIM)
    camera = np.array([np.cos(elev) * np.cos(azim), np.cos(elev) * np.sin(azim), np.sin(elev)])

    def move_to(theta):
        bob_x =  ell * np.sin(theta)
        bob_z = -ell * np.cos(theta)
        for rod, y_i, x, z in zip(rods, CLAMP, bob_x, bob_z):
            rod.set_data_3d([0, x], [y_i, y_i], [0, z])
        bobs.set_data_3d(bob_x, CLAMP, bob_z)

        # Each stripe turns with pendulum 1 in front of the first clamp, with pendulum 2 behind the second,
        # and twists smoothly from one to the other in between.
        twist_angle = np.interp(y, CLAMP, theta)
        for k, line in enumerate(stripe_lines):
            angle = twist_angle + 2 * np.pi * k / stripes
            x, z = RADIUS * np.sin(angle), -RADIUS * np.cos(angle)
            facing_us = x * camera[0] + z * camera[2] > 0
            line.set_data_3d(np.where(facing_us, x, np.nan), y, np.where(facing_us, z, np.nan))

    return move_to

With those, here's the sketch.

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(projection='3d')
draw_axle(ax)

theta = np.array([0.9, -0.9])
move_to = draw_pendulums(ax)
move_to(theta)

# Labels, and the straight-down direction that each angle is measured from
for i in range(2):
    ax.plot([0, 0], [CLAMP[i], CLAMP[i]], [0, -1.1], color='gray', linestyle='dashed', zorder=0)
    ax.text(np.sin(theta[i]) + 0.1, CLAMP[i], -np.cos(theta[i]) + 0.05, f'$\\theta_{i+1}$', fontsize=18, zorder=5)
for end in AXLE_ENDS:
    ax.text(0, end, -0.75, 'bearing', ha='center', fontsize=12)
ax.text(0, 0.5, 0.25, 'twisted axle', ha='center', color='C3', fontsize=14)
plt.show()

## The Torques

Each pendulum is a bob of mass $m_i$ on a light rod of length $\ell_i$, so its moment of inertia about the axle is $I_i = m_i \ell_i^2$.  (We'll let the two pendulums be different!)  Newton's second law for rotation says
\begin{align}
    I_i \frac{d^2\theta_i}{dt^2} &= \tau_i,
\end{align}
and each pendulum feels two torques: gravity, just like a single pendulum, and the twist of the axle,
\begin{align}
    \tau_1 &= -m_1 g \ell_1 \sin\theta_1 - \kappa (\theta_1 - \theta_2)
    \\
    \tau_2 &= -m_2 g \ell_2 \sin\theta_2 - \kappa (\theta_2 - \theta_1).
\end{align}
The twist pushes the two pendulums in opposite directions: it tries to bring them back into line.

Notice that the torques depend only on the angles, never on how fast anything is moving.  So velocity Verlet will be symplectic, just like it was for the pendulum, the Morse molecule, and the planets.

The potential energy is gravity's for each pendulum, plus the energy stored in the twist,
\begin{align}
    V &= m_1 g \ell_1 (1 - \cos\theta_1) + m_2 g \ell_2 (1 - \cos\theta_2) + \frac{1}{2}\kappa (\theta_1 - \theta_2)^2,
\end{align}
and each torque is minus the slope of $V$ in that pendulum's angle, $\tau_i = -\partial V / \partial \theta_i$.  The kinetic energy is $\frac{1}{2} I_1 \dot\theta_1^2 + \frac{1}{2} I_2 \dot\theta_2^2$.

Just like positions of many bodies had shape `(..., N, 3)`, our angles will have shape `(..., 2)`, where the last axis says *which* pendulum and the `...` are any batch axes.  So a state has shape `(2, ..., 2)`.  And just like for many bodies, the masses are back in, so we'll write our own `energy`.

In [ ]:
class CoupledPendulums(ConservativeSystem):
    '''
    Two pendulums clamped onto an axle that twists like a torsion spring.
    Pendulum i is a bob of mass m[i] on a light rod of length ell[i], and the axle between them
    has torsion constant kappa.  Angles and angular velocities have shape (..., 2).

    I_i d^2θ_i/dt^2 = τ_i = -m_i g ℓ_i sin(θ_i) - κ (θ_i - θ_other)
    '''

    def __init__(self, masses, lengths, kappa, g=9.8):
        self.m = np.array(masses, dtype=float)
        self.ell = np.array(lengths, dtype=float)
        self.kappa = kappa
        self.g = g
        self.I = self.m * self.ell**2   # each pendulum's moment of inertia about the axle

    def twist(self, x):
        return x[..., 0] - x[..., 1]

    def potential(self, x):
        gravity = np.sum(self.m * self.g * self.ell * (1 - np.cos(x)), axis=-1)
        return gravity + 0.5 * self.kappa * self.twist(x)**2

    def torque(self, x):
        gravity = -self.m * self.g * self.ell * np.sin(x)
        # The twist torque is -κ (θ_1 - θ_2) on pendulum 1 and the opposite on pendulum 2.
        twist = -self.kappa * self.twist(x)[..., np.newaxis] * np.array([1, -1])
        return gravity + twist

    def acceleration(self, t, x, v):
        return self.torque(x) / self.I

    def energy_of_each(self, x, v):
        # Each pendulum's own kinetic and gravitational energy, not counting the twist.  Shape (..., 2).
        return 0.5 * self.I * v**2 + self.m * self.g * self.ell * (1 - np.cos(x))

    def energy(self, x, v):
        return np.sum(self.energy_of_each(x, v), axis=-1) + 0.5 * self.kappa * self.twist(x)**2

## Trust, but Verify

Let's check that each torque really is minus the slope of the potential, nudging one angle at a time with a central difference.  This should print `True`.

In [ ]:
def gradient(function, x, Delta_x=1e-5):
    '''
    Estimates the gradient of a function of angles x, with shape (..., 2),
    with a central difference in each angle.
    '''
    result = np.zeros(np.shape(x))
    for i in range(2):
        step = np.zeros(2)
        step[i] = Delta_x
        result[..., i] = (function(x + step) - function(x - step)) / (2 * Delta_x)
    return result

rng = np.random.default_rng(320)

pendulums = CoupledPendulums(masses=[1.0, 2.0], lengths=[1.0, 0.5], kappa=3.0)
x = rng.normal(size=(100, 2))
print(np.allclose(pendulums.torque(x), -gradient(pendulums.potential, x)))

# Small Swings

Let's start gently, with two identical pendulums and a floppy axle, swinging only a little.  For small angles $\sin\theta \approx \theta$, and the torques become linear.

There are two especially simple ways the pair can move:
 - **Together**, with $\theta_1 = \theta_2$.  The axle never twists, so each pendulum swings as if it were alone, with angular frequency $\omega_\text{together} = \sqrt{g/\ell}$.
 - **Opposite**, with $\theta_2 = -\theta_1$.  Now the twist is $2\theta_1$, which adds an extra restoring torque $-2\kappa\theta_1$, so they swing faster, at $\omega_\text{opposite} = \sqrt{g/\ell + 2\kappa/(m\ell^2)}$.

These are the *normal modes*.  Every small motion is a mix of the two.

What if we pull back only pendulum 1, and let go?  Starting from $(\theta_0, 0)$ is half "together" plus half "opposite," since $(\theta_0, 0) = \frac{1}{2}(\theta_0, \theta_0) + \frac{1}{2}(\theta_0, -\theta_0)$, so
\begin{align}
    \theta_1(t) &= \frac{\theta_0}{2}\left(\cos\omega_\text{together} t + \cos\omega_\text{opposite} t\right)
    \\
    \theta_2(t) &= \frac{\theta_0}{2}\left(\cos\omega_\text{together} t - \cos\omega_\text{opposite} t\right).
\end{align}
The two frequencies are close, so the two cosines drift in and out of step.  When they're in step, pendulum 1 swings and pendulum 2 is still.  When they're out of step, it's the other way around!  They get completely out of step after a time $\pi / (\omega_\text{opposite} - \omega_\text{together})$.  These are *beats*, and the energy sloshes back and forth between the pendulums.

In [ ]:
m, ell, g = 1.0, 1.0, 9.8
floppy = CoupledPendulums(masses=[m, m], lengths=[ell, ell], kappa=0.05 * m * g * ell)

omega_together = np.sqrt(g / ell)
omega_opposite = np.sqrt(g / ell + 2 * floppy.kappa / (m * ell**2))
handoff = np.pi / (omega_opposite - omega_together)
print('ω together =', omega_together, '   ω opposite =', omega_opposite)
print('the energy should all move to pendulum 2 after', handoff, 'seconds')

theta_0 = 0.05
y_0 = np.array([[theta_0, 0.0], [0.0, 0.0]])   # angles, then angular velocities

Delta_t = 0.001
t, y = VelocityVerlet(floppy, Delta_t, round(2 * handoff / Delta_t)).starting_from(0, y_0)
print(y.shape)

In [ ]:
exact_1 = theta_0 / 2 * (np.cos(omega_together * t) + np.cos(omega_opposite * t))
exact_2 = theta_0 / 2 * (np.cos(omega_together * t) - np.cos(omega_opposite * t))

# Each pendulum's share of the energy the two pendulums have between them (leaving out the twist).
each = floppy.energy_of_each(y[:, 0], y[:, 1])
share = each / np.sum(each, axis=-1, keepdims=True)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(t, y[:, 0, 0], color='C0', label='$\\theta_1$')
axes[0].plot(t, y[:, 0, 1], color='C1', label='$\\theta_2$')
axes[0].plot(t, exact_1, color='black', linestyle='dashed', linewidth=0.8, label='small-angle exact')
axes[0].plot(t, exact_2, color='black', linestyle='dashed', linewidth=0.8)
axes[0].set_ylabel('angle')
axes[0].legend(loc='upper right')

axes[1].plot(t, share[:, 0], color='C0', label='pendulum 1')
axes[1].plot(t, share[:, 1], color='C1', label='pendulum 2')
total = floppy.energy(y[:, 0], y[:, 1])     # including the energy in the twist
axes[1].plot(t, total / total[0], color='black', label='total energy / starting energy')
axes[1].axvline(handoff, color='gray', linestyle='dotted')
axes[1].axvline(2 * handoff, color='gray', linestyle='dotted')
axes[1].set_xlabel('t (s)')
axes[1].set_ylabel('share of the energy')
axes[1].legend(loc='upper right')
plt.show()

print('largest difference from the small-angle exact solution, as a fraction of θ_0:', np.max(np.abs(y[:, 0, 0] - exact_1)) / theta_0)

The energy really does slosh completely from one pendulum to the other and back, right on schedule (the dotted lines).  And Verlet agrees with the small-angle exact solution; the leftover difference is mostly because $\sin\theta$ isn't *exactly* $\theta$.

## Out of Tune

That complete handoff happened because the two pendulums were identical.  Pendulum 2 was being pushed (through the axle) at exactly its own natural rhythm.  That's *resonance*, the same reason you pump a swing in time with how it already wants to go.

What if pendulum 2 is a little longer?  Then its natural frequency is a little lower, the pushes from pendulum 1 drift out of rhythm with it, and it can never soak up all the energy.  Lengths are parameters, so each choice is its own `CoupledPendulums`.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

for color, length_2 in zip(('C0', 'C1', 'C2', 'C3'), (1.0, 1.05, 1.1, 1.2)):
    detuned = CoupledPendulums(masses=[m, m], lengths=[ell, length_2], kappa=floppy.kappa)
    t, y = VelocityVerlet(detuned, Delta_t, round(3 * handoff / Delta_t)).starting_from(0, y_0)
    each = detuned.energy_of_each(y[:, 0], y[:, 1])
    share_1 = each[:, 0] / np.sum(each, axis=-1)
    ax.plot(t, share_1, color=color, label=f'$\\ell_2$ = {length_2} m')
    print(f'ℓ_2 = {length_2} m: pendulum 1 never has less than {share_1.min():.2f} of the energy')

ax.set_xlabel('t (s)')
ax.set_ylabel("pendulum 1's share of the energy")
ax.legend(loc='lower left')
plt.show()

The further out of tune, the less energy makes it across.  Everything here is still perfectly predictable, though.  Small swings make linear equations, and linear equations are tame.

# Big Swings: The Butterfly Effect

Now let's stiffen the axle and swing *hard*: both pendulums start nearly upside-down, on opposite sides.  At these angles $\sin\theta$ is nothing like $\theta$, so no more normal modes.

Here's the experiment.  We'll release *three* copies of the machine.  They're identical, except that in the second copy pendulum 1 starts a teeny bit further over, and in the third it starts a teeny bit less far over.  How teeny?  The code prints how far it moves the bob: about the width of a few atoms.  No real experiment could ever tell the two starts apart.

The three copies are just a batch of three, so the starting state has shape `(2, 3, 2)`: (angles or angular velocities) × (which copy) × (which pendulum).

In [ ]:
stiff = CoupledPendulums(masses=[m, m], lengths=[ell, ell], kappa=0.2 * m * g * ell)

start = np.array([2.8, -2.8])
nudge = 1e-9
print('the nudge moves the bob by', ell * nudge, 'meters')

angles = np.array([start, start + [nudge, 0], start - [nudge, 0]])
y_0 = np.array([angles, np.zeros_like(angles)])
print(y_0.shape)

T = 30
t, y = VelocityVerlet(stiff, Delta_t, round(T / Delta_t)).starting_from(0, y_0)
print(y.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, y[:, 0, 0, 0], color='C0', label='copy 1')
ax.plot(t, y[:, 0, 1, 0], color='C1', linestyle='dashed', label=f'copy 2, nudged by {nudge}')
ax.plot(t, y[:, 0, 2, 0], color='C2', linestyle='dotted', label=f'copy 3, nudged by {-nudge}')
ax.set_xlabel('t (s)')
ax.set_ylabel('$\\theta_1$')
ax.legend(loc='upper left')
plt.show()

For a while the three copies are indistinguishable; the dashed and dotted lines sit right on top of the solid one.  Then, suddenly, they go their separate ways and never come back together.

"Suddenly" is a little misleading, though.  Let's look at how far apart each pair of copies is, $d = \sqrt{(\Delta\theta_1)^2 + (\Delta\theta_2)^2}$, on a log scale.  With three copies there are three pairs.

In [ ]:
pairs = [(0, 1), (0, 2), (1, 2)]
styles = ['solid', 'dashed', 'dotted']    # so a curve hiding under another still shows

def separation_between(angles, a, b):
    '''How far apart copies a and b are, at every time.  angles has shape (steps+1, copies, 2).'''
    return np.sqrt(np.sum((angles[:, a] - angles[:, b])**2, axis=-1))

separations = [separation_between(y[:, 0], a, b) for a, b in pairs]

fig, ax = plt.subplots(figsize=(12, 4))
for (a, b), separation, style in zip(pairs, separations, styles):
    ax.plot(t, separation, linestyle=style, label=f'copies {a+1} and {b+1}')
ax.set_yscale('log')
ax.legend()
ax.set_xlabel('t (s)')
ax.set_ylabel('separation d (radians)')
plt.show()

(You can hardly see copies 1 and 2: they're hiding under copies 1 and 3.  The two nudges are equal and opposite, so at first those two pairs are exactly the same distance apart.  Copies 2 and 3 start twice as far apart.)

On a log scale each separation climbs along a (wobbly) straight line, until it's as big as the angles themselves and can't grow anymore.  A straight line on a log plot means *exponential* growth,
\begin{align}
    d(t) &\approx d(0)\, e^{\lambda t}.
\end{align}
The growth rate $\lambda$ is called the *Lyapunov exponent*.  After every time $1/\lambda$ the separation grows by another factor of $e$.

We can measure $\lambda$ by fitting a straight line to $\log d$ while it's growing: after the start has been forgotten, but before $d$ gets big.

In [ ]:
def fit_growth(t, separation):
    '''Fit a straight line to log(separation) while it's growing.  Returns the slope λ, the intercept, and where we fit.'''
    growing = (separation > 100 * nudge) & (separation < 1e-2)
    slope, intercept = np.polyfit(t[growing], np.log(separation[growing]), 1)
    return slope, intercept, growing

fig, ax = plt.subplots(figsize=(12, 4))
exponents = []
for (a, b), separation, style in zip(pairs, separations, styles):
    slope, intercept, growing = fit_growth(t, separation)
    exponents.append(slope)
    ax.plot(t, separation, linestyle=style, label=f'copies {a+1} and {b+1}')
    ax.plot(t[growing], np.exp(intercept + slope * t[growing]), color='black', linewidth=0.8)
    print(f'copies {a+1} and {b+1}:   λ = {slope} per second')

lyapunov = np.mean(exponents)
print('average Lyapunov exponent λ =', lyapunov, 'per second')
print('so the separation grows by a factor of 10 every', np.log(10) / lyapunov, 'seconds')

ax.set_yscale('log')
ax.set_xlabel('t (s)')
ax.set_ylabel('separation d (radians)')
ax.legend()
plt.show()

All three pairs separate at nearly the same rate, even though each pair started apart by a different amount and in a different direction.  The growth rate is a property of the *motion*, not of which nudge we picked.

Here's why that's devastating for prediction.  Suppose you want to know the angles to within 0.1 radian.  If your starting angles are off by $d(0)$, your prediction is good for about
\begin{align}
    t_\text{predict} &\approx \frac{1}{\lambda} \log\frac{0.1}{d(0)}.
\end{align}
Measure the start 10 times more precisely, and you only gain another $\log(10)/\lambda$ seconds.  A *billion* times more precisely only buys you about $\log(10^9)/\lambda$ seconds more.  Precision gets expensive exponentially fast, while the payoff only grows slowly.

## Is It Real?

Should we believe this?  Maybe the integrator is the one going haywire!  Two quick checks.

First, the energy.  Verlet is symplectic here, so it should hold the energy steady even while the copies fly apart.

Second, the step size.  If the exponential growth were a numerical artifact, it ought to change when we change $\Delta t$.  Let's halve it and measure $\lambda$ again.

In [ ]:
energy = stiff.energy(y[:, 0], y[:, 1])
print('largest relative change in energy, any copy:', np.max(np.abs(energy / energy[0] - 1)))

t_fine, y_fine = VelocityVerlet(stiff, Delta_t / 2, round(T / (Delta_t / 2))).starting_from(0, y_0)
exponents_fine = [fit_growth(t_fine, separation_between(y_fine[:, 0], a, b))[0] for a, b in pairs]
print('λ for each pair with Δt   =', np.array(exponents))
print('λ for each pair with Δt/2 =', np.array(exponents_fine))

The energy hardly budges, and the growth rate doesn't care about the step size.  The sensitivity is a property of the *physics*, not of our numerics.

There's a sobering flip side.  Every computer makes tiny roundoff errors, around $10^{-16}$, and in a chaotic system those grow at the same exponential rate as our nudge.  So after a while *no* computer can follow the exact trajectory from a given start, no matter how good the integrator.  What a good integrator still gets right is the physics: the energy, and the overall character of the motion.

## Not Every Start Is Chaotic

Is every big swing chaotic?  Let's run the same experiment from a slightly different start, with the pendulums a bit less far over, side by side with the first one.  That's a batch of 2 starts × 2 copies, so the angles have shape `(2, 2, 2)` and the whole state has shape `(2, 2, 2, 2)`.

In [ ]:
starts = np.array([[2.8, -2.8], [2.5, -2.5]])
copies = np.array([[0, 0], [nudge, 0]])

angles = starts[:, np.newaxis, :] + copies[np.newaxis, :, :]
print(angles.shape)
y_0 = np.array([angles, np.zeros_like(angles)])

t, y = VelocityVerlet(stiff, Delta_t, round(T / Delta_t)).starting_from(0, y_0)
print(y.shape)

separation = np.sqrt(np.sum((y[:, 0, :, 1] - y[:, 0, :, 0])**2, axis=-1))   # one separation per start
energy = stiff.energy(y_0[0, :, 0], y_0[1, :, 0])

fig, ax = plt.subplots(figsize=(12, 4))
for i, color in zip(range(len(starts)), ('C0', 'C2')):
    ax.plot(t, separation[:, i], color=color, label=f'start ({starts[i, 0]}, {starts[i, 1]}), E = {energy[i]:.1f} J')
ax.set_yscale('log')
ax.set_xlabel('t (s)')
ax.set_ylabel('separation d (radians)')
ax.legend()
plt.show()

From the second start, the nudge never takes off.  The separation creeps up a little, but nothing like exponentially, and the two copies stay together for the whole run.  That motion is *regular*, even though it has a similar energy and swings almost as hard.

That's typical.  At low energies, like our small swings, everything is regular.  At high energies, regular and chaotic motion are tangled together, and which one you get depends on exactly where you start.

# Watch It

Let's watch the two copies of our first, chaotic start, from the same angle as the sketch.  The stripes on the axle twist along with copy 1.  Copy 2 is drawn faintly, right underneath copy 1, so for a while you'll only see one machine.  Then the butterfly flaps its wings.

The movie plays in real time.

In [ ]:
import imageio_ffmpeg
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Point matplotlib at the copy of ffmpeg that came with `uv sync`.
plt.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()

theta = y[:, 0, 0]   # the chaotic start: shape (steps+1, 2 copies, 2 pendulums)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(projection='3d')
draw_axle(ax, top=1.2)   # leave room above the axle, for swings over the top

# Copy 2 is drawn first, faintly, so it ends up underneath copy 1.  The stripes show copy 1's twist.
move_copy_2 = draw_pendulums(ax, alpha=0.35, color='C1', stripes=0)
move_copy_1 = draw_pendulums(ax)
clock = ax.set_title('')

def draw(step):
    move_copy_1(theta[step, 0])
    move_copy_2(theta[step, 1])
    clock.set_text(f't = {t[step]:.2f} s')

# Show every frame_step-th step: 40 steps of 1 ms is 0.04 s per frame, 25 frames per second.
frame_step = 40
animation = FuncAnimation(fig, draw, frames=range(0, len(t), frame_step), interval=1000 * frame_step * Delta_t)
plt.close(fig)   # show only the movie, not a leftover still frame
HTML(animation.to_html5_video())

That's chaos: a perfectly deterministic machine, with no randomness anywhere, whose future we can't predict for long because we can never know its present well enough.

What did it take?  Not randomness, and not velocity-dependent forces.  Just two ingredients: *nonlinear* forces (the $\sin\theta$s), and enough room to move.  A single pendulum can't be chaotic; its phase space is too cramped.  Two coupled pendulums can.